# Project01 · RAG + Gemini v2

Code thuật toán đầy đủ nằm trong các cell bên dưới. 01: dữ liệu/index/KG/retrieval và inference CDC; 02: generation/agents/Verifier/API; 03: evaluation/analysis. Các import liên notebook dùng backend export của chính các cell này; không đọc implementation từ provenance hoặc folder nhóm. Sửa source tại notebook, chạy `scripts/llm_v2_phase2/notebooks.py export`; không sửa backend riêng.

Giữ attribution project RAG/KG của nhóm; bảng nguồn/hàm/cell ở docs/llm_v2/reuse_map.json và artifacts/llm_v2/phase2/actual_reuse.json. Môi trường: WSL Python3.11 serving riêng, không cài vào env Torch/Keras. Mặc định RUN_API/RUN_EVALUATION/RUN_TRAINING/RUN_TEST đều False, ALLOW_PAID_API=False. Không Run All để train/TEST lịch sử.

Kernel chạy source hiện tại cần các dependency serving đã khóa. Nguồn MedQuAD lịch sử không thay lời khuyên lâm sàng hiện hành.

In [1]:
from pathlib import Path
import sys,types,__future__
from IPython.core.magic import register_cell_magic
PROJECT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'docs/llm_v2/provider_config.example.json').exists())
sys.path.insert(0,str(PROJECT/'web/backend'))
RUN_API=False
RUN_EVALUATION=False
RUN_TRAINING=False
RUN_TEST=False
ALLOW_PAID_API=False
@register_cell_magic
def llm_v2_module(line,cell):
    name=line.strip();parent,_,child=name.rpartition('.')
    if parent not in sys.modules:
        package=types.ModuleType(parent);package.__path__=[str(PROJECT/'web/backend'/parent)];sys.modules[parent]=package
    module=sys.modules.get(name)
    if module is None:
        module=types.ModuleType(name);module.__package__=parent;module.__file__=str(PROJECT/'web/backend'/Path(*name.split('.')).with_suffix('.py'))
        sys.modules[name]=module;setattr(sys.modules[parent],child,module)
    exec(compile(cell,'<project01-llm-v2-notebook>/'+name,'exec',flags=__future__.annotations.compiler_flag,dont_inherit=True),module.__dict__)


## evaluation

Input → xử lý → output theo docstring và từng hàm. Các đoạn source port giữ tên class/hàm của nhóm; extension cho Gemini/medical được ghi trong bản đồ port.

In [2]:
%%llm_v2_module project01_llm_v2.evaluation
"""Single-claim held-out evaluation and saved-result analysis; never training."""



In [3]:
%%llm_v2_module project01_llm_v2.evaluation
from pathlib import Path



In [4]:
%%llm_v2_module project01_llm_v2.evaluation
from datetime import datetime,timezone



In [5]:
%%llm_v2_module project01_llm_v2.evaluation
import asyncio,hashlib,json,os,random,time



### read_rows

In [6]:
%%llm_v2_module project01_llm_v2.evaluation
def read_rows(path):return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines()]



### file_hash

In [7]:
%%llm_v2_module project01_llm_v2.evaluation
def file_hash(path):
    with Path(path).open('rb') as f:return hashlib.file_digest(f,'sha256').hexdigest()



### save_json

In [8]:
%%llm_v2_module project01_llm_v2.evaluation
def save_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temporary=path.with_suffix('.tmp');temporary.write_text(json.dumps(value,ensure_ascii=False,indent=2)+'\n',encoding='utf-8');temporary.replace(path)



### resume_branch

In [9]:
%%llm_v2_module project01_llm_v2.evaluation
async def resume_branch(runtime,variant,question,call_prefix,profile=None,progress=None,deadline=None):
    from .provider import ProviderUnavailable
    deadline=deadline or runtime.gateway.config['queue']['comparison_deadline_seconds']
    started=time.monotonic();usage_start=len(runtime.gateway.observations);minute_retries=0
    while True:
        try:
            remaining=deadline-(time.monotonic()-started)
            if remaining<=0:raise ProviderUnavailable('timeout','Đã hết thời gian pipeline.')
            if variant=='llm_direct':task=runtime.direct(question,call_prefix)
            else:task=runtime.rag(question,call_prefix,profile,progress)
            result=await asyncio.wait_for(task,remaining)
            result['usage']=runtime.gateway.observations[usage_start:]
            result['latency_ms']=(time.monotonic()-started)*1000
            result['minute_retries']=minute_retries
            return result
        except ProviderUnavailable as exc:
            if exc.code=='waiting_minute' and minute_retries<2 and time.monotonic()-started+61<deadline:
                minute_retries+=1
                if progress:progress('Đợi quota theo phút · giữ bước đã hoàn tất')
                await asyncio.sleep(60)
                continue
            raise



### planned_order

In [10]:
%%llm_v2_module project01_llm_v2.evaluation
def planned_order(questions):
    generator=random.Random(20261102);output=[]
    for index,case in enumerate(questions):
        variants=['llm_direct','agentic_rag']
        if index%2:variants.reverse()
        output.extend((case,variant) for variant in variants)
    generator.shuffle(output)
    return output



### evaluate

In [11]:
%%llm_v2_module project01_llm_v2.evaluation
async def evaluate(root,max_new_outputs=None):
    from .agents import AgenticRuntime
    from .provider import ProviderUnavailable
    root=Path(root);phase=root/'artifacts/llm_v2/phase2'
    freeze=json.loads((phase/'freeze.json').read_text())
    if freeze['stage']!='HELD_OUT_FROZEN':raise ValueError('No approved freeze')
    for record in freeze['files']:
        if file_hash(root/record['path'])!=record['sha256']:raise ValueError('Frozen input drift: '+record['path'])
    questions=read_rows(root/'datasets/llm_v2/evaluation_v1/held_out_questions.jsonl')
    assert len(questions)==40
    folder=phase/'evaluation';folder.mkdir(parents=True,exist_ok=True)
    lock=folder/'runner.lock'
    import fcntl
    with lock.open('a') as handle:
        try:fcntl.flock(handle,fcntl.LOCK_EX|fcntl.LOCK_NB)
        except BlockingIOError:raise RuntimeError('One evaluation runner only')
        runtime=AgenticRuntime(root,allow_api=True);new=0
        for case,variant in planned_order(questions):
            identifier=case['id']+'_'+variant;output=folder/'results'/(identifier+'.json');claim=folder/'claims'/(identifier+'.json')
            if output.exists():continue
            if max_new_outputs is not None and new>=max_new_outputs:break
            state=json.loads(claim.read_text()) if claim.exists() else {'id':identifier,'freeze_id':freeze['freeze_id'],'created_at':datetime.now(timezone.utc).isoformat(),'state':'claimed'}
            if state['freeze_id']!=freeze['freeze_id']:raise ValueError('Claim freeze mismatch')
            if state['state'] in {'unknown','failed','complete'}:raise ValueError('Consumed claim without final output; require incident revision')
            save_json(claim,state)
            attempt_started=time.monotonic();usage_start=len(runtime.gateway.observations)
            def accumulate_usage():
                known={r['call_id']:r for r in state.get('usage',[])}
                for observation in runtime.gateway.observations[usage_start:]:
                    cid=observation['call_id']
                    if cid not in known or observation.get('cache_status')=='live':known[cid]=observation
                state['usage']=list(known.values())
                state['elapsed_active_ms']=state.get('elapsed_active_ms',0)+(time.monotonic()-attempt_started)*1000
            try:
                result=await resume_branch(runtime,variant,case['question_vi'],'evaluation:'+freeze['freeze_id']+':'+identifier)
                result['freeze_id']=freeze['freeze_id']
                models={r.get('model_returned') for r in result['usage'] if r.get('kind')=='generation'}
                if models!={freeze['model_returned']}:raise ValueError('Provider model/version drift')
                state['state']='complete'
            except ProviderUnavailable as exc:
                if exc.code in {'waiting_quota','waiting_minute','claim_waiting_quota'}:
                    accumulate_usage();state['state']='waiting_quota';state['reason']=exc.code;save_json(claim,state)
                    save_json(folder/'checkpoint.json',{'status':'WAITING_QUOTA','next_claim':identifier,'new_outputs_this_run':new,'models_changed':False})
                    return {'status':'WAITING_QUOTA','new_outputs':new,'next_claim':identifier}
                state['state']='failed';state['reason']=exc.code
                result={'variant':variant,'question':case['question_vi'],'status':'provider_error','answer_vi':'','error_code':exc.code,'usage':[],
                        'latency_ms':None,'freeze_id':freeze['freeze_id']}
            except (ValueError,asyncio.TimeoutError) as exc:
                state['state']='unknown';state['reason']=type(exc).__name__
                result={'variant':variant,'question':case['question_vi'],'status':'provider_error','answer_vi':'','error_code':type(exc).__name__,
                        'usage':[],'latency_ms':None,'freeze_id':freeze['freeze_id']}
            accumulate_usage();result['usage']=state.get('usage',[])
            result['latency_ms']=state['elapsed_active_ms']
            result['calendar_elapsed_ms']=(datetime.now(timezone.utc)-datetime.fromisoformat(state['created_at'])).total_seconds()*1000
            result['quota_pause_ms']=max(0,result['calendar_elapsed_ms']-result['latency_ms'])
            save_json(output,{'question_id':case['id'],'variant':variant,'repeat':1,'category':case['category'],'result':result,'created_at':datetime.now(timezone.utc).isoformat()})
            save_json(claim,state);new+=1
        count=len(list((folder/'results').glob('*.json')))
        status='OUTPUTS_COMPLETE_REVIEW_PENDING' if count==80 else 'IN_PROGRESS'
        save_json(folder/'checkpoint.json',{'status':status,'outputs':count,'new_outputs_this_run':new})
        return {'status':status,'outputs':count,'new_outputs':new}



### blinded_review_pack

In [12]:
%%llm_v2_module project01_llm_v2.evaluation
def blinded_review_pack(root):
    root=Path(root);folder=root/'artifacts/llm_v2/phase2/evaluation'
    rows=[json.loads(p.read_text()) for p in sorted((folder/'results').glob('*.json'))]
    by_question={}
    for row in rows:by_question.setdefault(row['question_id'],[]).append(row)
    public=[];mapping=[]
    for i,(qid,group) in enumerate(sorted(by_question.items())):
        group=sorted(group,key=lambda r:r['variant'],reverse=bool(i%2))
        for label,row in zip(['A','B'],group):
            public.append({'question_id':qid,'label':label,'question':row['result']['question'],'answer_vi':row['result']['answer_vi'],
                           'scores':{'focus':None,'correctness':None,'completeness':None,'limits':None},'critical_error':None,'reviewer':None})
            mapping.append({'question_id':qid,'label':label,'variant':row['variant']})
    save_json(folder/'review_pack.json',{'scope':'assistant/manual review pending, branch names hidden; citations may reveal RAG','rows':public})
    save_json(folder/'review_mapping.json',mapping)
    return len(public)



### summarize_saved

In [13]:
%%llm_v2_module project01_llm_v2.evaluation
def summarize_saved(root):
    root=Path(root);folder=root/'artifacts/llm_v2/phase2/evaluation'
    rows=[json.loads(p.read_text()) for p in sorted((folder/'results').glob('*.json'))]
    summary=[]
    for variant in ['llm_direct','agentic_rag']:
        selected=[r for r in rows if r['variant']==variant]
        summary.append({'variant':variant,'outputs':len(selected),'expected':40,
                        'complete':sum(r['result']['status']=='complete' for r in selected),
                        'errors':sum(r['result']['status']=='provider_error' for r in selected),
                        'quality_score':None,'quality_review':'pending','paid_budget_usd':0})
    return {'status':'not_complete' if len(rows)<80 else 'outputs_complete_review_pending','rows':summary,
            'historical_results_reused':False,'clinical_accuracy_claimed':False}



## Kết quả mới đã lưu

40 câu×2 nhánh, một lần mỗi câu/nhánh sau freeze; lỗi vẫn trong mẫu số. Không dùng P0/P1/P2/82,5% cũ. Chất lượng phải được review, không suy từ HTTP200/độ dài câu trả lời. Chưa đủ80 output thì báo chưa xong, không tạo biểu đồ chất lượng giả.

In [14]:
from project01_llm_v2.evaluation import summarize_saved
display(summarize_saved(PROJECT))


{'status': 'not_complete',
 'rows': [{'variant': 'llm_direct',
   'outputs': 0,
   'expected': 40,
   'complete': 0,
   'errors': 0,
   'quality_score': None,
   'quality_review': 'pending',
   'paid_budget_usd': 0},
  {'variant': 'agentic_rag',
   'outputs': 0,
   'expected': 40,
   'complete': 0,
   'errors': 0,
   'quality_score': None,
   'quality_review': 'pending',
   'paid_budget_usd': 0}],
 'historical_results_reused': False,
 'clinical_accuracy_claimed': False}

## Review và biểu đồ từ saved outputs
Code dưới chỉ đọc80 output và review sau freeze. Không sinh điểm khi review còn thiếu. quality: điểm0–8; evidence: tỉ lệ hỗ trợ/Recall@6; latency_calls: giây vàSDK call mới, cache tách riêng. Không gọi LLM/TEST/train.

In [15]:
%%llm_v2_module project01_llm_v2.analysis
"""Review/metrics/plots from saved80 outputs only; no inference or tuning."""



In [16]:
%%llm_v2_module project01_llm_v2.analysis
from pathlib import Path



In [17]:
%%llm_v2_module project01_llm_v2.analysis
import json,math,statistics



In [18]:
%%llm_v2_module project01_llm_v2.analysis
def load(path):return json.loads(Path(path).read_text(encoding='utf-8'))



In [19]:
%%llm_v2_module project01_llm_v2.analysis
def wilson_interval(successes,n):
    if not n:return None
    z=1.96;p=successes/n;den=1+z*z/n
    middle=(p+z*z/(2*n))/den
    radius=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return [max(0,middle-radius),min(1,middle+radius)]



In [20]:
%%llm_v2_module project01_llm_v2.analysis
def score_saved_reviews(root,review_file):
    root=Path(root);folder=root/'artifacts/llm_v2/phase2/evaluation'
    raw=[load(p) for p in sorted((folder/'results').glob('*.json'))]
    if len(raw)!=80:raise ValueError('Need all80 outputs, including failures, before final scoring')
    mapping={(r['question_id'],r['label']):r['variant'] for r in load(folder/'review_mapping.json')}
    reviews=load(review_file)['rows'];review_by_key={}
    for row in reviews:
        key=(row['question_id'],mapping[(row['question_id'],row['label'])])
        if key in review_by_key or not row.get('reviewer'):raise ValueError('Missing reviewer or duplicate review')
        values=[row['scores'][name] for name in ['focus','correctness','completeness','limits']]
        if not all(type(v) is int and 0<=v<=2 for v in values) or type(row.get('critical_error')) is not bool:raise ValueError('Incomplete rubric')
        review_by_key[key]=row
    if len(review_by_key)!=80:raise ValueError('Need80 reviewed outputs')
    references=[json.loads(line) for line in (root/'datasets/llm_v2/evaluation_v1/held_out_reference.jsonl').read_text().splitlines()]
    gold={r['id']:r for r in references};scored=[]
    for row in raw:
        key=(row['question_id'],row['variant']);review=review_by_key[key];result=row['result'];points=sum(review['scores'].values())
        success=points>=6 and review['scores']['correctness']==2 and not review['critical_error']
        qrels={cid for ref in gold[row['question_id']]['reference_evidence'] if ref['relevance']==2 for cid in ref['relevant_chunk_ids']}
        ranked=[e['chunk_id'] for e in result.get('evidence',[])[:6]]
        matches=[i for i,cid in enumerate(ranked,1) if cid in qrels]
        citation_reviews=review.get('citation_reviews',[])
        if row['variant']=='agentic_rag' and result.get('claims'):
            expected={c['id'] for c in result['claims']}
            if {r['claim_id'] for r in citation_reviews}!=expected:raise ValueError('Review every emitted medical claim separately')
        supported=sum(r['status']=='supported' for r in citation_reviews)
        live=[r for r in result.get('usage',[]) if r.get('cache_status')=='live']
        tokens=[r['usage']['total_tokens'] for r in live if r.get('usage') and r['usage'].get('total_tokens') is not None]
        scored.append({'question_id':row['question_id'],'variant':row['variant'],'rubric_total':points,'success':success,
          'critical_error':review['critical_error'],'reviewer':review['reviewer'],'status':result['status'],
          'recall_at6':len(set(ranked)&qrels)/len(qrels) if qrels and row['variant']=='agentic_rag' else None,
          'mrr_at6':1/matches[0] if matches else (0 if qrels and row['variant']=='agentic_rag' else None),
          'citation_supported':supported,'citation_reviewed':len(citation_reviews),
          'latency_seconds':result.get('latency_ms',0)/1000 if result.get('latency_ms') is not None else None,
          'live_sdk_calls':len(live),'provider_tokens_live':sum(tokens) if tokens else None,'missing_usage_calls':len(live)-len(tokens)})
    summary=[]
    for variant in ['llm_direct','agentic_rag']:
        rows=[r for r in scored if r['variant']==variant];n=len(rows);wins=sum(r['success'] for r in rows)
        latencies=[r['latency_seconds'] for r in rows if r['latency_seconds'] is not None]
        q=[r for r in rows if r['recall_at6'] is not None]
        support=sum(r['citation_supported'] for r in rows);claims=sum(r['citation_reviewed'] for r in rows)
        summary.append({'variant':variant,'n':n,'success_count':wins,'task_success':wins/n,'wilson95':wilson_interval(wins,n),
          'rubric_mean':statistics.mean(r['rubric_total'] for r in rows),'p50_seconds':statistics.median(latencies) if latencies else None,
          'latency_n':len(latencies),'citation_support':support/claims if claims else None,'citation_support_numerator':support,
          'citation_support_denominator':claims,'recall_at6':statistics.mean(r['recall_at6'] for r in q) if q else None,
          'retrieval_n':len(q),'live_sdk_calls':sum(r['live_sdk_calls'] for r in rows),'free_cost_configured_usd':0})
    pairs={qid:{r['variant']:r['rubric_total'] for r in scored if r['question_id']==qid} for qid in gold}
    paired={'rag_win':sum(r['agentic_rag']>r['llm_direct'] for r in pairs.values()),'tie':sum(r['agentic_rag']==r['llm_direct'] for r in pairs.values()),
            'direct_win':sum(r['agentic_rag']<r['llm_direct'] for r in pairs.values()),'n':40}
    report={'rows':scored,'summary':summary,'paired':paired,'review_scope':'reviewers as named, not presumed clinical experts',
            'qrels_complete':False,'historical_scores_reused':False,'cache_and_live_separated':True}
    output=folder/'scored_summary.json'
    if output.exists():raise ValueError('Scored report exists; use a versioned review revision')
    output.write_text(json.dumps(report,ensure_ascii=False,indent=2)+'\n')
    return report



In [21]:
%%llm_v2_module project01_llm_v2.analysis
def plot_saved_report(report,output_folder):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    folder=Path(output_folder);folder.mkdir(parents=True,exist_ok=True)
    rows=report['summary'];labels=['LLM trực tiếp','Đa agent + RAG'];colors=['#8796ad','#2857ed']
    fig,ax=plt.subplots(figsize=(7,4.5));ax.bar(labels,[r['rubric_mean'] for r in rows],color=colors);ax.set_ylim(0,8)
    ax.set_ylabel('Điểm rubric trung bình (0–8)');ax.set_title('Chất lượng trên40 câu hỏi cùng cặp')
    fig.tight_layout();fig.savefig(folder/'quality.png',dpi=180);plt.close(fig)
    rag=next(r for r in rows if r['variant']=='agentic_rag')
    fig,ax=plt.subplots(figsize=(7,4.5));names=['Claim có hỗ trợ','Recall@6 theo qrels'];values=[rag['citation_support'],rag['recall_at6']]
    for i,value in enumerate(values):
        if value is not None:ax.bar(i,value,color='#2857ed')
        else:ax.text(i,.05,'N/A',ha='center')
    ax.set_xticks(range(2),names);ax.set_ylim(0,1);ax.set_ylabel('Tỷ lệ (0–1)');ax.set_title('Chứng cứ của RAG · qrels chưa đầy đủ')
    fig.tight_layout();fig.savefig(folder/'evidence.png',dpi=180);plt.close(fig)
    fig,axes=plt.subplots(1,2,figsize=(9,4.5))
    axes[0].bar(labels,[r['p50_seconds'] or 0 for r in rows],color=colors);axes[0].set_ylabel('Giây');axes[0].set_title('Latency p50 toàn pipeline')
    axes[1].bar(labels,[r['live_sdk_calls'] for r in rows],color=colors);axes[1].set_ylabel('SDK calls mới');axes[1].set_title('Tổng overhead API')
    fig.tight_layout();fig.savefig(folder/'latency_calls.png',dpi=180);plt.close(fig)
    (folder/'FIGURE_GUIDE.md').write_text('''# Cách đọc ba hình

quality.png: trục ngang là hai nhánh, trục dọc điểm rubric trung bình0–8 trên cùng40 câu; điểm cao hơn chỉ phản ánh bộ benchmark nhỏ được review, không là độ chính xác lâm sàng.

evidence.png: chỉ dành cho RAG. Cột đầu là claim được người review xác nhận có evidence hỗ trợ / số claim đã review; cột sau là Recall@6 theo qrels có sẵn trên nhóm câu answerable. Qrels chưa exhaustive; N/A không được biến thành100%.

latency_calls.png: hình trái là p50 giây toàn pipeline, gồm chờ/retry trong phiên; thời gian chờ reset nhiều ngày ghi riêng ở raw result. Hình phải là tổng SDK call live của40 câu, cache tách riêng; không phải số HTTP request nếu SDK có hành vi mạng phụ. Free-tier cost cấu hình0USD, không là xác nhận hóa đơn.
''',encoding='utf-8')

